<a href="https://colab.research.google.com/github/marcnadeau/psychic-octo-broccoli/blob/main/nb/Llama3_(8B)-Ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Playlist LoRA training

Ce notebook prépare le dataset Spotify local, entraîne un adaptateur LoRA avec Unsloth sur un runtime Colab GPU, puis sauvegarde l'adaptateur. Les étapes Ollama et GGUF sont volontairement séparées du training.

### Installation

In [ ]:
# Installation adaptative et idempotente des dépendances requises
# Exécuter cette cellule, puis redémarrer le kernel si des paquets ont été installés.
import importlib
import subprocess
import sys

# Use stable, compatible versions
required = {
    "unsloth": "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
    "transformers": "transformers>=4.57.0",
    "trl": "trl>=0.22.0",
    "datasets": "datasets>=3.4.1",
    "bitsandbytes": "bitsandbytes",
    "accelerate": "accelerate",
    "peft": "peft",
    "sentencepiece": "sentencepiece",
}

# First, try importing to see what's missing
to_install = []
for mod, pkg in required.items():
    try:
        importlib.import_module(mod)
    except Exception as e:
        to_install.append(pkg)

if to_install:
    print("Installation/mise à jour des paquets...")
    # Clean install: uninstall old versions first to avoid conflicts
    old_packages = ["unsloth", "unsloth_zoo"]
    for pkg in old_packages:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", pkg], 
                       capture_output=True)
    
    # Install fresh
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + to_install
    subprocess.check_call(cmd)
    print("✓ Installation terminée. Redémarre le kernel maintenant.")
    print("\nREDÉMARRE LE KERNEL (important: Ctrl+M R ou Kernel > Restart)")
else:
    print("✓ Toutes les dépendances requises sont déjà présentes.")

In [ ]:
# Unsloth must be imported before transformers, peft, or torch.
import unsloth
from unsloth import FastLanguageModel

import importlib
import traceback
import sys
import torch

print("Unsloth imported with optimizations enabled.")
print("CUDA count:", torch.cuda.device_count())


Installation des paquets manquants: ['unsloth', 'unsloth_zoo==2026.8.15', 'trl==0.22.2', 'bitsandbytes']
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 890.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

### Unsloth

In [5]:
from unsloth import FastLanguageModel
import torch
print("CUDA count:", torch.cuda.device_count())
print("Param device:", next(model.parameters()).device)
# Assure-toi aussi que les inputs sont envoyés sur le même device (trainer le gère normalement).

# -----------------------------
# 1. Charger le modèle Qwen 2.5 Instruct en 4-bit
# -----------------------------
# Change uniquement cette valeur pour tester un autre modèle compatible Unsloth.
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# -----------------------------
# 2. Préparer LoRA
# -----------------------------
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1531: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


ImportError: cannot import name 'is_torch_neuron_available' from 'transformers.utils' (/usr/local/lib/python3.12/dist-packages/transformers/utils/__init__.py)

<a name="Data"></a>
### Data preparation

The JSONL files are generated locally from the Spotify Million Playlist Dataset and use the columns `instruction`, `input`, and `output`.

In [ ]:
from pathlib import Path

from datasets import Dataset, load_dataset


def find_jsonl(filename):
    """Find a dataset file in Kaggle and local notebook locations."""
    roots = [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
        Path.cwd() / "dataset",
        Path.cwd(),
    ]
    matches = []
    for root in roots:
        if root.exists():
            matches.extend(root.rglob(filename))

    unique_matches = list(dict.fromkeys(path.resolve() for path in matches))
    if not unique_matches:
        raise FileNotFoundError(
            f"Impossible de trouver {filename}. "
            "Ajoute le dataset Kaggle ou vérifie son nom de fichier."
        )
    if len(unique_matches) > 1:
        print(f"Plusieurs fichiers {filename} trouvés; utilisation de: {unique_matches[0]}")
    return unique_matches[0]


train_path = find_jsonl("train.jsonl")
validation_path = find_jsonl("validation.jsonl")
print(f"Train: {train_path}")
print(f"Validation: {validation_path}")

raw_train = load_dataset("json", data_files=str(train_path), split="train")
raw_validation = load_dataset("json", data_files=str(validation_path), split="train")


def prepare_dataset(data):
    examples = []
    for row in data:
        if "text" in row and row["text"]:
            examples.append({"text": row["text"]})
            continue

        instr = row.get("instruction", "") or row.get("prompt", "") or row.get("query", "")
        extra_input = row.get("input", "") or ""
        out = row.get("output", "") or row.get("response", "") or row.get("answer", "")

        if extra_input and extra_input not in instr:
            instr = (instr + "\nYour input is:\n" + extra_input).strip() if instr else extra_input

        if not instr and not out:
            continue

        text = f"### Instruction:\n{instr.strip()}\n\n### Response:\n{out.strip()}\n"
        examples.append({"text": text})

    return Dataset.from_list(examples)


dataset = prepare_dataset(raw_train)
validation_dataset = prepare_dataset(raw_validation)

print("Train examples:", len(dataset))
print("Validation examples:", len(validation_dataset))
print(dataset.column_names)
print(dataset[0]["text"][:600])


<a name="Train"></a>
### Train the model

Le trainer démarre avec 60 steps pour vérifier rapidement le pipeline. Après ce test, remplace `max_steps=60` par `max_steps=-1` et ajoute `num_train_epochs=1` pour un entraînement complet.

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=validation_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        logging_steps=1,
        eval_strategy="steps",
        eval_steps=30,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("qwen25_lora")  # Sauvegarde locale de l'adaptateur
tokenizer.save_pretrained("qwen25_lora")
# model.push_to_hub("votre_nom/qwen25_lora", token="VOTRE_HF_TOKEN")
# tokenizer.push_to_hub("votre_nom/qwen25_lora", token="VOTRE_HF_TOKEN")

### Export local optionnel

Le dossier `llama_lora` contient seulement l'adaptateur LoRA. Pour l'utiliser dans une application locale, exporte plutôt un fichier GGUF.

In [ ]:
from pathlib import Path
import shutil

# Kaggle lit dans /kaggle/input et écrit dans /kaggle/working.
# Colab utilise /content.
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists():
    export_dir = Path("/kaggle/working/playlist_qwen25_gguf")
elif "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules:
    export_dir = Path("/content/playlist_qwen25_gguf")
else:
    export_dir = Path.cwd() / "playlist_qwen25_gguf"

if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained_gguf(
    str(export_dir),
    tokenizer,
    quantization_method="q4_k_m",
)

print(f"GGUF sauvegardé dans : {export_dir}")
print([path.name for path in export_dir.iterdir()])